# AgentCore Runtime: the minimum viable deploy (Strands)

**Goal of this notebook:** one agent, written once, running in four places, with nothing else provisioned.

| Stage | Where it runs | Command |
|---|---|---|
| 1. In-process test | Your Python kernel | a cell in this notebook |
| 2. Local server | `localhost:8080` on your laptop | `agentcore dev` |
| 3. Cloud | AgentCore Runtime (AWS) | `agentcore deploy` |
| 4. Called from code | Anywhere with AWS credentials | `boto3` / Strands |

**Nothing else is created.** No Memory resource, no Gateway, no Lambda, no Identity provider, no Knowledge Base, no Guardrail. Those are separate primitives with separate costs. Runtime alone is the smallest thing that proves "my agent is deployed and answering".

```mermaid
flowchart LR
    CODE[main.py<br/>your agent] --> CLI[agentcore CLI]
    CLI --> CDK[AWS CDK / CloudFormation]
    CDK --> RT[AgentCore Runtime<br/>HTTPS endpoint + ARN]
    CALLER[boto3 / CLI / another agent] --> RT
    RT --> BR[Amazon Bedrock<br/>model call]
```

## 1. Why the old notebook broke

Two different tools have owned the `agentcore` command name.

| | Starter Toolkit (old) | AgentCore CLI (current) |
|---|---|---|
| Package | `pip install bedrock-agentcore-starter-toolkit` | `npm install -g @aws/agentcore` |
| Commands | `configure` / `launch` / `invoke` | `create` / `dev` / `deploy` / `invoke` |
| Project shape | `.bedrock_agentcore.yaml` beside your file | `agentcore/` + `app/<AgentName>/` |
| Status | Deprecated, superseded | The one AWS points new work at |

Both install a binary called `agentcore`. If the old one is still on your PATH, uninstall it first:

```bash
pip uninstall bedrock-agentcore-starter-toolkit
# or: pipx uninstall bedrock-agentcore-starter-toolkit
# or: uv tool uninstall bedrock-agentcore-starter-toolkit
```

The old notebooks wrote a `.py` file into a scratch folder and told you to run `agentcore configure -e file.py`. That path no longer matches the project layout the current CLI creates, which is why the file you had stopped lining up with the tool.

## 2. The exact cause of `ModuleNotFoundError: No module named 'bedrock_agentcore'`

This is not a bug in your code. It is a half-built virtual environment, and the CLI is designed to start the server anyway.

```mermaid
flowchart TD
    A[agentcore dev] --> B{app/MyAgent/.venv exists<br/>and has uvicorn?}
    B -- yes --> F[start uvicorn]
    B -- no --> C[uv venv]
    C --> D[uv sync<br/>reads app/MyAgent/pyproject.toml]
    D -- success --> F
    D -- FAILS --> E[fallback: uv pip install uvicorn ONLY]
    E --> F
    F --> G[uvicorn imports runtime_agent:app]
    G --> H[import bedrock_agentcore --> ModuleNotFoundError]
```

Read the red box. When `uv sync` fails, the HTTP dev server does not stop. It installs bare `uvicorn` and starts anyway. The server comes up, the browser inspector opens, and the very first import in your file blows up. The traceback you saw points at line 2 of your agent because that is the first line that needs a package the venv never got.

**Three facts that follow from this:**

| Fact | Consequence |
|---|---|
| Dependencies come from `app/<AgentName>/pyproject.toml` | Adding an import to your `.py` file installs nothing |
| The venv lives at `app/<AgentName>/.venv` | Your notebook kernel's packages are irrelevant to it |
| `uv sync` failure is non-fatal for `dev` | A green-looking server can be broken inside |

The fix is one command run in the right directory, and this notebook does it for you in section 7.

## 3. Prerequisites

| Requirement | Check | If missing |
|---|---|---|
| Node.js 20+ | `node -v` | nodejs.org |
| `uv` | `uv --version` | `curl -LsSf https://astral.sh/uv/install.sh \| sh` |
| Python 3.10+ | `python3 -V` | python.org |
| AgentCore CLI | `agentcore --version` | `npm install -g @aws/agentcore` |
| AWS credentials | `aws sts get-caller-identity` | `aws configure` |
| Bedrock model access | the ping cell below | Bedrock console, us-east-1 |

**IAM reality check.** `agentcore deploy` drives AWS CDK and CloudFormation. The caller needs CloudFormation, S3, IAM `CreateRole` and `PassRole` for `*BedrockAgentCore*` roles, plus `bedrock-agentcore:*`. A learner account scoped to Bedrock model access alone cannot run the deploy step. Sections 1 through 8 (local) will still work for them; section 9 onwards needs the deploy-capable role.

In [ ]:
# Preflight. Nothing here changes anything, it only reports.
import json, os, re, shutil, subprocess, sys, textwrap, uuid, pathlib

def sh(cmd, cwd=None, timeout=600, quiet=False):
    """Run a shell command, stream nothing, return (code, combined output)."""
    p = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True, timeout=timeout)
    out = ((p.stdout or "") + (p.stderr or "")).strip()
    if not quiet:
        print(f"$ {cmd}")
        print(out[:4000] if out else "(no output)")
        print("-" * 70)
    return p.returncode, out

checks = {
    "node":      "node -v",
    "npm":       "npm -v",
    "uv":        "uv --version",
    "python":    "python3 -V",
    "agentcore": "agentcore --version",
    "aws cli":   "aws --version",
}
print("TOOL CHECK")
for name, cmd in checks.items():
    code, out = sh(cmd, quiet=True)
    first = out.splitlines()[0] if out else ""
    print(f"  {'OK ' if code == 0 else 'MISSING'}  {name:<10} {first[:60]}")

In [ ]:
# Identity, region, and a real model call. This is the cheapest way to prove
# the credentials in this kernel can actually reach Bedrock.
import boto3

REGION   = "us-east-1"
MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"   # 'us.' prefix is mandatory for on-demand Claude

try:
    ident = boto3.client("sts", region_name=REGION).get_caller_identity()
    ACCOUNT = ident["Account"]
    print("caller :", ident["Arn"])
    print("account:", ACCOUNT, "| region:", REGION)
except Exception as e:
    ACCOUNT = None
    print("STS failed. Configure credentials before going further:", e)

try:
    br = boto3.client("bedrock-runtime", region_name=REGION)
    r = br.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "Reply with the single word: ready"}]}],
        inferenceConfig={"maxTokens": 10},
    )
    print("model  :", r["output"]["message"]["content"][0]["text"].strip())
except Exception as e:
    print("Bedrock call failed:", type(e).__name__, str(e)[:300])

## 4. What `agentcore create` actually builds

```
MyFirstRuntimeAgent/          <- project root, all agentcore commands run from here
├── agentcore/
│   ├── agentcore.json        <- WHAT to deploy: runtimes, memories, credentials
│   ├── aws-targets.json      <- WHERE to deploy: account + region
│   ├── .env.local            <- API keys for non-Bedrock providers (gitignored)
│   └── cdk/                  <- generated CDK app, you rarely touch this
├── app/
│   └── MyAgent/              <- one directory per agent
│       ├── main.py           <- the entrypoint file
│       ├── pyproject.toml    <- the ONLY place dependencies count
│       ├── model/load.py     <- model provider wiring
│       └── .venv/            <- created by uv, used by `agentcore dev`
└── README.md
```

Two files decide everything, and they are the two people get wrong:

| File | Answers | Failure if wrong |
|---|---|---|
| `agentcore/agentcore.json` | which file is the entrypoint | dev server imports the wrong module |
| `app/<Agent>/pyproject.toml` | what gets installed | `ModuleNotFoundError` |

**You already have this project.** If you want to start clean, this is the exact non-interactive command:

```bash
agentcore create \
  --project-name MyFirstRuntimeAgent \
  --name MyAgent \
  --language Python \
  --framework Strands \
  --model-provider Bedrock \
  --protocol HTTP \
  --build CodeZip \
  --memory none
```

Watch out for `--defaults` on its own. In current CLI versions that flag creates a **harness** project (config-driven, no agent code), which is a different teaching story. Passing `--framework` keeps you on the code-based path.

In [ ]:
# Point at your existing project. Change PROJECT_DIR if yours lives elsewhere.
PROJECT_DIR = pathlib.Path(
    "/Users/akash-at-work/Documents/IBS Agentic AI and AWS GenAI Training/"
    "Day-11 - AgentCore/demos/MyFirstRuntimeAgent"
).expanduser()

print("project:", PROJECT_DIR)
print("exists :", PROJECT_DIR.exists())

if PROJECT_DIR.exists():
    for p in sorted(PROJECT_DIR.rglob("*")):
        rel = p.relative_to(PROJECT_DIR)
        parts = rel.parts
        if any(x in parts for x in (".venv", "node_modules", ".git", "__pycache__", "cdk.out")):
            continue
        if len(parts) > 3:
            continue
        print(("  " * (len(parts) - 1)) + ("[dir] " if p.is_dir() else "      ") + parts[-1])

In [ ]:
# Read the project's own config instead of assuming file names.
CFG_PATH = PROJECT_DIR / "agentcore" / "agentcore.json"
cfg = json.loads(CFG_PATH.read_text()) if CFG_PATH.exists() else {}

runtimes = cfg.get("runtimes", [])
print(f"runtimes declared: {len(runtimes)}")
for rt in runtimes:
    print(json.dumps({k: v for k, v in rt.items() if not isinstance(v, (dict, list))}, indent=2))

RT = runtimes[0] if runtimes else {}
AGENT_NAME  = RT.get("name", "MyAgent")
ENTRYPOINT  = RT.get("entrypoint", "main.py")            # e.g. 'main.py' or 'runtime_agent.py'
CODE_LOC    = RT.get("codeLocation", f"app/{AGENT_NAME}")
AGENT_DIR   = (PROJECT_DIR / CODE_LOC).resolve()
AGENT_FILE  = AGENT_DIR / ENTRYPOINT.split(":")[0]
PYPROJECT   = AGENT_DIR / "pyproject.toml"

print("\nresolved:")
print("  agent name :", AGENT_NAME)
print("  agent dir  :", AGENT_DIR)
print("  entrypoint :", AGENT_FILE.name, "->", "uvicorn module:", AGENT_FILE.stem + ":app")
print("  pyproject  :", PYPROJECT, "exists" if PYPROJECT.exists() else "MISSING")

targets = PROJECT_DIR / "agentcore" / "aws-targets.json"
if targets.exists():
    print("\naws-targets.json:", targets.read_text().strip()[:400])
    print("\nIf the region here is not us-east-1, edit this file before deploying.")

## 5. The contract your file has to satisfy

AgentCore Runtime is a container host with two HTTP endpoints. It has no opinion about your framework.

| Endpoint | Method | Who calls it | What it must do |
|---|---|---|---|
| `/invocations` | POST | callers, through `InvokeAgentRuntime` | take a JSON payload, return a result |
| `/ping` | GET | the platform | report health |

The `bedrock_agentcore` SDK gives you both for free. You write one decorated function.

```mermaid
flowchart LR
    P["payload: {prompt: ...}"] --> E["@app.entrypoint<br/>invoke(payload, context)"]
    E --> A[Strands Agent loop]
    A --> M[Bedrock model]
    A --> T[your @tool functions]
    E --> R["return dict --> JSON"]
```

**Return type decides the wire format, and this trips people up:**

| Your entrypoint returns | Response content type | Parse it with |
|---|---|---|
| a `dict` | `application/json` | `json.loads(body)` |
| a generator / async generator | `text/event-stream` | read `data: {...}` lines |

We return a dict. It is the shorter demo and the simpler parse. Streaming is a one-function change, shown at the end.

### Where to build the Agent object: pick deliberately

| Pattern | Behaviour | Use when |
|---|---|---|
| One global `Agent` at import | History from every caller piles into one object | Never in a multi-user demo |
| Per-session cache keyed by `context.session_id` | Each conversation keeps its own history in memory, lost on cold start | Default for chat, no extra AWS resources |
| New `Agent` per invocation | Stateless, no cross-talk, no memory | The minimum, and what we use here |

We build the **model and tools once** at import (cheap to share, safe) and the **Agent per call** (stateless). Durable memory across restarts is a separate primitive, deliberately out of scope.

In [ ]:
# Write the agent. This overwrites whatever entrypoint file the config points at.
AGENT_SRC = '''"""Minimum AgentCore Runtime agent: Strands + Amazon Bedrock.

Contract: POST /invocations with {"prompt": "..."} -> {"result": "..."}
Run locally:  agentcore dev
Deploy:       agentcore deploy
"""

import json
import os

from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel

MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
REGION = os.environ.get("AWS_REGION", os.environ.get("AWS_DEFAULT_REGION", "us-east-1"))

SYSTEM_PROMPT = (
    "You are TravelMind, an airline support agent. "
    "Use get_pnr for any question about a booking. "
    "Answer in three sentences or fewer. Never invent a booking."
)

# --- built once at import: cheap to share, no per-caller state ---
app = BedrockAgentCoreApp()
model = BedrockModel(model_id=MODEL_ID, region_name=REGION)

_BOOKINGS = {
    "JX48Q2": {"passenger": "Rao", "tier": "Gold",
               "segment": "BLR-DEL", "status": "CANCELLED"},
}


@tool
def get_pnr(pnr: str) -> str:
    """Look up an airline booking by its PNR code."""
    return json.dumps(_BOOKINGS.get(pnr.strip().upper(), {"error": "PNR not found"}))


@app.entrypoint
def invoke(payload, context):
    """Entrypoint. Validate input, run one stateless agent turn, return JSON."""
    prompt = payload.get("prompt") if isinstance(payload, dict) else None
    if not isinstance(prompt, str) or not prompt.strip():
        return {"error": "payload must contain a non-empty 'prompt' string"}

    agent = Agent(model=model, tools=[get_pnr], system_prompt=SYSTEM_PROMPT)
    result = agent(prompt)

    return {
        "result": str(result),
        "session_id": getattr(context, "session_id", None),
        "model": MODEL_ID,
    }


if __name__ == "__main__":
    app.run()
'''

AGENT_DIR.mkdir(parents=True, exist_ok=True)
AGENT_FILE.write_text(AGENT_SRC)
print("wrote", AGENT_FILE, f"({len(AGENT_SRC)} bytes)")
print("uvicorn will import:", AGENT_FILE.stem + ":app")

### Read the file you just wrote

| Line | Why it is there |
|---|---|
| `app = BedrockAgentCoreApp()` | Creates the Starlette app with `/invocations` and `/ping`. `agentcore dev` runs `uvicorn <file>:app`, so the variable must be named `app` |
| `region_name=REGION` | AgentCore Runtime sets `AWS_REGION` in the container. Locally it falls back to `us-east-1` |
| `us.` model prefix | On-demand Claude on Bedrock requires the cross-region inference profile. A bare model id raises `ValidationException` |
| `@tool` docstring | Strands sends the docstring to the model as the tool description. A vague docstring is a vague tool |
| the `isinstance` guard | The payload is whatever the caller posted. Validate before handing it to a model |
| `str(result)` | `AgentResult.__str__` gives the text. `result.message` is a dict of content blocks and needs digging |
| `if __name__ == "__main__"` | Lets you run `python main.py` directly, outside the CLI |

In [ ]:
# The file that actually controls the venv. Everything your agent imports goes here.
PYPROJECT_SRC = f'''[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "{AGENT_NAME.lower()}"
version = "0.1.0"
description = "AgentCore Runtime application using the Strands SDK"
requires-python = ">=3.10"
dependencies = [
    "aws-opentelemetry-distro",
    "bedrock-agentcore>=1.9.1",
    "botocore[crt]>=1.35.0",
    "strands-agents>=1.15.0",
]

[tool.hatch.build.targets.wheel]
packages = ["."]
'''

PYPROJECT.write_text(PYPROJECT_SRC)
print("wrote", PYPROJECT)
print(PYPROJECT_SRC)

| Dependency | What breaks without it |
|---|---|
| `bedrock-agentcore` | `ModuleNotFoundError` on line 1, the error you hit. It also pulls in `uvicorn` and `starlette` |
| `strands-agents` | The agent framework |
| `aws-opentelemetry-distro` | Traces stop reaching CloudWatch GenAI Observability |
| `botocore[crt]` | Slower and less reliable transport for large payloads |

Deliberately absent: `strands-agents-tools`, `mcp`, memory integrations. Add them the day you use them, and only then.

In [ ]:
# THE FIX for ModuleNotFoundError. Run uv sync in the agent directory, and read the error if it fails.
code, out = sh("uv sync", cwd=str(AGENT_DIR), timeout=900)
print("uv sync exit code:", code)

if code != 0:
    print("\nuv sync FAILED. `agentcore dev` would still start a server on a broken venv.")
    print("Common causes: requires-python not satisfiable, no network or proxy blocking PyPI,")
    print("a malformed pyproject.toml, or a package with no wheel for your Python version.")

In [ ]:
# Prove the venv is complete. This is the check the CLI does not do for you.
venv_py = AGENT_DIR / ".venv" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
print("venv python:", venv_py, "exists" if venv_py.exists() else "MISSING")

if venv_py.exists():
    probe_src = (
        "import importlib.util\n"
        "for m in ['bedrock_agentcore', 'strands', 'uvicorn', 'boto3', 'opentelemetry']:\n"
        "    found = importlib.util.find_spec(m) is not None\n"
        "    print(f\"  {m:<20} {'OK' if found else 'MISSING'}\")\n"
    )
    probe_path = AGENT_DIR / "_probe.py"
    probe_path.write_text(probe_src)
    print("packages visible to the dev server:")
    sh(f'"{venv_py}" "{probe_path}"', quiet=False)
    probe_path.unlink(missing_ok=True)

## 6. Stage 1: run the agent in this kernel, with no server at all

`@app.entrypoint` returns your function unchanged. It stays an ordinary Python function, so you can call it directly. This is the fastest debug loop available: no port, no container, no HTTP, full tracebacks.

The one thing to supply is a `context` object with a `session_id`.

In [ ]:
# In-process test. Uses the venv python so it exercises the SAME packages the dev server will use.
test_prompt = "My flight BLR to DEL was cancelled. PNR JX48Q2. What is my status?"

harness = f'''
import json, sys
sys.path.insert(0, {str(AGENT_DIR)!r})
from {AGENT_FILE.stem} import invoke

class Ctx:
    session_id = "local-test-" + "0" * 24

out = invoke({{"prompt": {test_prompt!r}}}, Ctx())
print(json.dumps(out, indent=2)[:2000])

bad = invoke({{"nope": 1}}, Ctx())
print("\\nguard rail check:", bad)
'''
harness_path = AGENT_DIR / "_local_test.py"
harness_path.write_text(harness)
code, out = sh(f'"{venv_py}" "{harness_path}"', cwd=str(AGENT_DIR), timeout=300)
harness_path.unlink(missing_ok=True)

## 7. Stage 2: the local server and the agent inspector

This one needs a terminal. `agentcore dev` starts a long-lived server and opens a browser, so a notebook cell would hang forever.

**Terminal A, from the project root:**

```bash
cd "<your project root>"
agentcore dev
```

What that does, in order:

1. finds or creates `app/MyAgent/.venv`
2. runs `uv sync` (the step you already did, so it is instant now)
3. runs `.venv/bin/uvicorn main:app --reload --host 127.0.0.1 --port 8080`
4. opens the agent inspector in your browser

**Terminal B, same project root:**

```bash
agentcore dev "What is the status of PNR JX48Q2?"
agentcore dev "What is the status of PNR JX48Q2?" --stream
curl -X POST http://localhost:8080/invocations \
  -H "Content-Type: application/json" \
  -d '{"prompt":"What is the status of PNR JX48Q2?"}'
curl http://localhost:8080/ping
```

| Flag | Use |
|---|---|
| `-p 3000` | Port 8080 already taken |
| `--logs` | Print server logs to stdout instead of the browser UI |
| `-b` | Terminal chat UI instead of the browser inspector |
| `--skip-deploy` | Do not touch AWS before starting |

`--reload` is on, so editing `main.py` restarts the server. Editing `pyproject.toml` does **not** reinstall anything. That needs `uv sync` again.

```mermaid
flowchart TD
    subgraph LOCAL[Your laptop]
        UV[uv venv + uv sync] --> VENV[app/MyAgent/.venv]
        VENV --> U[uvicorn main:app --reload :8080]
        U --> INS[agent inspector in browser]
    end
    U -->|boto3 with your credentials| BR[Amazon Bedrock<br/>real model calls, real cost]
```

The local server is local. The **model call is not.** `agentcore dev` still bills Bedrock tokens against your account.

## 8. Stage 3: deploy

```mermaid
flowchart LR
    A[agentcore deploy] --> B[uv pip install --target staging<br/>aarch64 wheels only]
    B --> C[zip code + deps]
    C --> D[upload to CDK S3 bucket]
    D --> E[CDK synth --> CloudFormation]
    E --> F[IAM execution role]
    E --> G[AgentCore Runtime + DEFAULT endpoint]
    E --> H[CloudWatch log group]
```

| What is created | Where to find it |
|---|---|
| CloudFormation stack | CloudFormation console, named after the project |
| IAM execution role | IAM, search `BedrockAgentCore` |
| AgentCore Runtime + ARN | `agentcore status` |
| Log group | `/aws/bedrock-agentcore/runtimes/<agent-id>-DEFAULT` |
| Code zip | The CDK staging bucket in S3 |

Two things worth knowing before you press go:

- **ARM64.** Runtime is Graviton. The packager installs `aarch64` wheels with `--only-binary :all:`. A dependency with no ARM wheel for your Python version fails the build. Fix by pinning `pythonVersion` in `agentcore.json` to a version with wheels, usually `PYTHON_3_12`.
- **CDK bootstrap.** First deploy in an account and region needs a one-time `CDKToolkit` stack. The CLI detects it and offers to do it, which means the first deploy is better run in a terminal where you can answer the prompt.

**Run the first deploy in a terminal:**

```bash
agentcore deploy
```

Once bootstrap exists, the cell below is safe to run from the notebook on every later deploy.

In [ ]:
# Non-interactive deploy. First run takes a few minutes; later runs are faster.
# If this hangs or asks for bootstrap, stop and run `agentcore deploy` once in a terminal.
code, deploy_out = sh("agentcore deploy -y", cwd=str(PROJECT_DIR), timeout=3600)
print("\ndeploy exit code:", code)

In [ ]:
# Find the runtime ARN. Parsed out of status rather than hand-copied.
code, status_out = sh("agentcore status --json", cwd=str(PROJECT_DIR), timeout=300, quiet=True)

arns = sorted(set(re.findall(r"arn:aws[\w-]*:bedrock-agentcore:[^\"'\s,]+runtime/[^\"'\s,]+", status_out)))
print("runtime ARNs found:", len(arns))
for a in arns:
    print("  ", a)

AGENT_ARN = arns[0] if arns else None
if not AGENT_ARN:
    print("\nNo ARN yet. Run the deploy cell, then `agentcore status` in a terminal to see why.")
    print(status_out[:1500])

In [ ]:
# Invoke through the CLI. Session id must be at least 33 characters.
SESSION_ID = "ibs-day11-demo-" + uuid.uuid4().hex   # 15 + 32 = 47 chars

sh(
    'agentcore invoke --prompt "What is the status of PNR JX48Q2 and who is the passenger?" '
    f'--session-id {SESSION_ID}',
    cwd=str(PROJECT_DIR),
    timeout=600,
)

## 9. Stage 4: invoke it from code

Three doors into the same runtime.

| Door | Auth | Use when |
|---|---|---|
| `agentcore invoke` | Your AWS profile | Demos, smoke tests, CI |
| `boto3` `invoke_agent_runtime` | SigV4 from AWS credentials | Your backend calls the agent |
| Direct HTTPS POST | OAuth bearer token | The caller is a user or a partner system, not an AWS principal |

The rules that bite:

- `runtimeSessionId` must be **33 characters or more**. Reuse it to continue a conversation, change it to start fresh.
- The response body is `application/json` for a dict return and `text/event-stream` for a generator. The helper below handles both, so it keeps working when you switch to streaming.
- The caller needs `bedrock-agentcore:InvokeAgentRuntime` on the runtime ARN.

In [ ]:
# A reusable client. Handles both response shapes.
acr = boto3.client("bedrock-agentcore", region_name=REGION)

def parse_body(raw: bytes):
    text = raw.decode("utf-8", errors="replace").strip()
    if "data:" in text:                       # server-sent events from a streaming entrypoint
        events = []
        for line in text.splitlines():
            line = line.strip()
            if line.startswith("data:"):
                chunk = line[5:].strip()
                try:
                    events.append(json.loads(chunk))
                except json.JSONDecodeError:
                    events.append(chunk)
        return {"stream_events": events}
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"raw": text}

def ask(prompt: str, session_id: str, arn: str = None):
    resp = acr.invoke_agent_runtime(
        agentRuntimeArn=arn or AGENT_ARN,
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}).encode("utf-8"),
        qualifier="DEFAULT",
    )
    raw = b"".join(chunk for chunk in resp.get("response", []))
    return parse_body(raw)

if AGENT_ARN:
    answer = ask("What is the status of PNR JX48Q2?", SESSION_ID)
    print(json.dumps(answer, indent=2)[:1500])
else:
    print("Deploy first, then re-run the ARN cell.")

In [ ]:
# Session behaviour, made visible. Same id vs new id.
if AGENT_ARN:
    same = ask("And who is the passenger on it?", SESSION_ID)          # same session
    fresh = ask("And who is the passenger on it?",
                "fresh-session-" + uuid.uuid4().hex)                    # new session

    print("SAME SESSION :", str(same)[:400])
    print()
    print("FRESH SESSION:", str(fresh)[:400])
    print()
    print("This agent builds a stateless Agent per call, so neither reply remembers the PNR.")
    print("The session id still isolates traces and would key AgentCore Memory if you added it.")

### Calling the deployed agent from another agent

A deployed runtime is just an HTTPS endpoint behind an ARN. Wrap it in a Strands `@tool` and any other agent can delegate to it. This is the honest version of "multi-agent": one process owns the conversation, another owns a capability, and the boundary between them is an API call you can log, throttle, and version.

In [ ]:
# Client-side agent that treats the deployed runtime as one tool.
from strands import Agent as ClientAgent, tool as client_tool
from strands.models import BedrockModel as ClientBedrockModel

@client_tool
def travelmind_runtime(question: str) -> str:
    """Ask the deployed TravelMind booking agent about a PNR or a flight status."""
    if not AGENT_ARN:
        return json.dumps({"error": "runtime not deployed"})
    out = ask(question, "delegated-" + uuid.uuid4().hex)
    return json.dumps(out)[:4000]

if AGENT_ARN:
    supervisor = ClientAgent(
        model=ClientBedrockModel(model_id=MODEL_ID, region_name=REGION),
        tools=[travelmind_runtime],
        system_prompt=(
            "You are a front desk assistant. For anything about a booking, call "
            "travelmind_runtime and summarise what it returns in one sentence."
        ),
    )
    print(supervisor("A passenger is asking about PNR JX48Q2. What should I tell them?"))

## 10. Observability

Runtime emits session metrics to CloudWatch on its own. Spans need one extra thing, done once per account and region: enable **CloudWatch Transaction Search**. Without it, the GenAI Observability page stays empty and people conclude tracing is broken.

| Question | Command |
|---|---|
| What did it print? | `agentcore logs` |
| What did it print an hour ago, errors only? | `agentcore logs --since 1h --level error` |
| What did it do internally? | `agentcore traces list` then `agentcore traces get <id>` |
| Where are the raw logs? | CloudWatch, `/aws/bedrock-agentcore/runtimes/<agent-id>-DEFAULT` |

The `aws-opentelemetry-distro` dependency plus the CLI's `enableOtel` setting mean the entrypoint is launched under `opentelemetry-instrument`. Traces flow without any code in your agent.

In [ ]:
sh("agentcore logs --since 30m -n 40", cwd=str(PROJECT_DIR), timeout=300)

In [ ]:
sh("agentcore traces list", cwd=str(PROJECT_DIR), timeout=300)

## 11. Where this fails

The failures that actually happen, and what each one looks like.

| Symptom | Real cause | Fix |
|---|---|---|
| `ModuleNotFoundError` on import | `uv sync` failed, dev server fell back to installing bare `uvicorn` | Run `uv sync` in `app/<Agent>/` and read the error |
| Import works locally, fails after deploy | The package is in your kernel, not in `pyproject.toml` | Dependencies live in `app/<Agent>/pyproject.toml` only |
| Deploy fails during dependency install | No `aarch64` wheel for the pinned Python version | Set `pythonVersion` to `PYTHON_3_12` in `agentcore.json` |
| `ValidationException` on the model id | Missing `us.` inference profile prefix | Use `us.anthropic.claude-haiku-4-5-...` |
| `AccessDeniedException` on `bedrock:InvokeModel` | Execution role lacks the inference profile ARN | Grant `InvokeModel` on the profile ARN and `foundation-model/*`. `bedrock:Converse` is not a valid IAM action |
| First deploy stalls with a prompt | CDK bootstrap missing | Run `agentcore deploy` once in a terminal and accept |
| `agentcore` runs the wrong tool | Old starter toolkit still installed | Uninstall it, both use the same binary name |
| Port 8080 busy | Another dev server, or a stale process | `agentcore dev -p 3000` |
| Session id rejected | Under 33 characters | Use a UUID with a prefix |
| Deploy denied | Caller lacks CloudFormation, S3, or IAM `CreateRole` | Deploy from a role that has them, not a Bedrock-only lab user |

**The cost question, since somebody always asks.** Runtime bills for the time your agent is actually running, and Bedrock bills tokens separately. A cancelled demo that leaves a deployed runtime idle is cheap. A Memory resource, a Gateway, and an OpenSearch collection left running are not. That asymmetry is the argument for the minimum deploy.

## 12. Optional upgrade: streaming

One change to the entrypoint turns the response into server-sent events, so tokens appear as they are produced instead of after the full turn.

```python
@app.entrypoint
async def invoke(payload, context):
    prompt = payload.get("prompt", "")
    agent = Agent(model=model, tools=[get_pnr], system_prompt=SYSTEM_PROMPT)
    async for event in agent.stream_async(prompt):
        if isinstance(event, dict) and "event" in event:
            yield event
```

| Changes | Stays the same |
|---|---|
| `def` becomes `async def` with `yield` | The file, the CLI commands, the ARN |
| Response is `text/event-stream` | `agentcore invoke --stream` now streams |
| `json.loads(body)` breaks | The `parse_body` helper above already handles it |

Teach the dict version first. Streaming is a delivery choice, not a capability.

## 13. Cleanup

`remove all` clears the local config, and the deploy that follows tells CloudFormation to tear the resources down. Removing the local files without the second command leaves the AWS resources running.

```bash
agentcore remove all
agentcore deploy
```

In [ ]:
# Uncomment both lines when the class is over.
# sh("agentcore remove all", cwd=str(PROJECT_DIR), timeout=600)
# sh("agentcore deploy -y", cwd=str(PROJECT_DIR), timeout=3600)
print("Cleanup is commented out on purpose. Uncomment when the demo is finished.")

## 14. What we deliberately left out

Every one of these was in the old notebook. Each is a real AWS resource with its own lifecycle, and none is needed to prove a deployed agent works.

| Primitive | Add it when | Cost of adding it early |
|---|---|---|
| Memory | The agent must remember across turns or sessions | Provisioning wait, eventual consistency, a resource to delete |
| Gateway | Many tools, or tools shared across agents, or an authed backend | OAuth setup, a Lambda, a target schema |
| Identity | The agent acts on a user's behalf against a third party | A credential provider before anything runs |
| Code Interpreter | The agent must compute, not assert | A sandbox session per call |
| Browser | The target site has no API | A browser sandbox and Playwright |
| Knowledge Base | Answers must be grounded in your documents | A vector store and an ingestion pipeline |
| Guardrails | Input or output needs policy enforcement | A guardrail version to manage |

**The order to teach them in:** get one agent deployed and answering, then add exactly one primitive and show what it changed. A demo that provisions seven things at once cannot tell you which one broke.

**Next:** the same four-command loop with no framework at all, then the same loop with LangChain and LangGraph. The agent code changes. The project layout, the commands, and the invoke path do not.